# RLS Management Functions

## Summary — what each function does and how they interact

| Function | What it does | Used for |
|---|---|---|
| `sanitize_role_name` | Strips characters not allowed in role names (commas, periods, etc.) | Called by every function that creates or looks up role names — **single source of truth** for naming |
| `build_dax_filters` | Builds the DAX filter expression for a role from config + value + global_filters. Supports `extra_filter` (fixed AND condition, e.g. DocType) and `filter_template` (custom expression, e.g. CONTAINSSTRING) | Used by `create_or_replace_roles` |
| `preprocess_concatenated_roles` | Combines several columns into a synthetic "concatenated" role | ⚠️ Must run BEFORE any other function if the config contains `concatenated_from` — returns an enriched `df`/`config` that must be used everywhere after |
| `create_or_replace_roles` | Creates/updates roles and their DAX filters in the semantic model. Supports `full_access`, `consolidated`, `consolidated_default`, concatenated roles, and standard roles | **Active role-creation function** |
| `add_members_to_roles` | Adds members from scratch (does not remove or sync departures) | Initial load / one-off use — does not handle removals |
| `drop_unused_roles` | Removes roles whose value no longer exists in the source RLS table | Cleanup after config or data changes |
| `remove_all_members_from_roles` | Clears all members of the managed roles, without dropping the roles | Step 1 of `sync_members_to_roles` |
| `sync_members_to_roles` | Clears + reloads all members from scratch | Full reset — expensive on roles with many members |
| `update_members_delta` | Syncs only the delta (adds/removes). `chunk_size` configurable (defaults to `1`, replicating the original one-by-one behavior) | **Daily/recurring execution** |
| `snapshot_current_membership` | Saves a full snapshot (role, user) of current membership | Before and after every sync run |
| `snapshot_run_summary` | One-row-per-run summary (environment, model, date, before/after/delta) | Quick monitoring — spotting anomalous variations |
| `log_membership_changes` | Persists one row per actual add/remove between two snapshots | Granular audit trail of exactly what changed |
| `run_with_audit` | Wraps any sync function with full audit (snapshots, change log, summary, alerts) | **Recommended entry point** for any membership sync |

## Interactions to keep in mind

- `create_or_replace_roles` must run **before** any membership function.
- `sync_members_to_roles` and `update_members_delta` **must not run in parallel** against the same dataset.
- If you use `preprocess_concatenated_roles`, the resulting `df`/`config` must be passed to **every** function that follows.
- `drop_unused_roles` relies on `source_column` matching the real RLS table — running it against a config out of sync with the data can drop valid roles.
- `consolidated_default` requires the keys listed in `exclude_special_keys` to exist in the same `config`.
- `chunk_size=1` by default in `update_members_delta` — does not change production behavior unless explicitly raised.
- Concatenated role sources that are numeric columns (e.g. `DocType`) must set `"is_numeric": True` in their `concatenated_from` definition, or the generated DAX will compare an Integer column against a Text literal and fail at query time.

## Special role types (`special`)

| `special` | Membership | When to use it |
|---|---|---|
| `full_access` | `source_column == source_value` | Full access, no DAX filter |
| `consolidated` | `source_column == source_value` **and** every column not listed in `ignore_cols` is empty | Opt-in role |
| `consolidated_default` | **Everyone** except those eligible for the keys in `exclude_special_keys` | Baseline filter for everyone, with explicit exceptions |


### Setup

In [ ]:
env = 'prod'
workspace_current = 'Manual exec'

In [ ]:
import sempy_labs as labs
from sempy_labs.tom import connect_semantic_model

## Functions

#### sanitize_role_name

In [ ]:
import re

def sanitize_role_name(value: str) -> str:
    """
    Strips/replaces characters not allowed in Tabular Object Model role names.
    Disallowed: , ; : ' " | * ? & / \ < > + = ( ) { } [ ] ! @ # $ % ^ . and whitespace.
    """
    value = str(value)
    value = re.sub(r'[,;:\'"|*?&/\\<>+=(){}\[\]!@#$%^.]', '', value)
    value = re.sub(r'\s+', '_', value.strip())
    value = re.sub(r'_+', '_', value)  # collapse repeated underscores
    return value

#### DAX helper

In [ ]:
# ── HELPER: Build DAX filters per table ───────────────────────────────────────
def build_dax_filters(settings, value, global_filters):
    """
    - Builds the role's DAX filter for a specific value
    - Supports 'extra_filter' (fixed AND-ed condition, e.g. "[DocType] = 7")
    - Supports 'filter_template' (custom expression, e.g. CONTAINSSTRING for multi-valued fields)
    - Merges with global_filters, grouped per table with &&
    """
    filters_by_table = {}

    role_column     = settings.get("column")
    extra_filter    = settings.get("extra_filter")
    filter_template = settings.get("filter_template")

    if role_column:
        table = settings["table"]
        if filter_template:
            value_expr = filter_template.format(value=value)
        else:
            value_expr = f'[{role_column}] = "{value}"'
        if extra_filter:
            value_expr = f"({extra_filter}) && ({value_expr})"
        filters_by_table.setdefault(table, []).append(value_expr)

    for gf in global_filters:
        table = gf["table"]
        expr = f'[{gf["column"]}] = "{gf["value"]}"'
        filters_by_table.setdefault(table, []).append(expr)

    return [
        (table, " && ".join(f"({expr})" for expr in exprs))
        for table, exprs in filters_by_table.items()
    ]

#### Preprocess functions -> Concatenated roles

In [ ]:
# ── CONCATENATED ROLES PRE-PROCESSING ─────────────────────────────────────────
# Call BEFORE any other role-management function.
# Enriches dataframe and config so that all functions below work unchanged.
#
# Config entry — two modes:
#   Mode 1 (list): "prefix": "...", "concatenated_from": ["ColA", "ColB"]
#   Mode 2 (dict):  "concatenated_from": {"ColA": {"table":..., "column":..., "is_numeric": False}, ...}
#
# "is_numeric": True marks a source column as numeric (e.g. DocType), so the
# generated DAX compares it WITHOUT quotes ([DocType] = 7) instead of as text
# ([DocType] = "7") — required whenever the target column's data type in the
# model is Integer/Whole Number, or DAX raises a type-comparison error.
# ─────────────────────────────────────────────────────────────────────────────

import pyspark.sql.functions as F
from pyspark.sql import DataFrame
from itertools import combinations


def _resolve_concatenated_sources(cat_key, cat_settings, config):
    cf = cat_settings["concatenated_from"]

    if isinstance(cf, dict):
        for src_key, src_def in cf.items():
            if "table" not in src_def or "column" not in src_def:
                raise ValueError(
                    f"[{cat_key}] Mode 2 source '{src_key}' must define both 'table' and 'column'."
                )
        return cf

    resolved = {}
    for src_key in cf:
        if src_key not in config:
            raise ValueError(
                f"[{cat_key}] concatenated_from key '{src_key}' not found in config. "
                f"Either add it as a standard config entry or switch to Mode 2 (dict)."
            )
        src_settings = config[src_key]
        if src_settings.get("special"):
            raise ValueError(
                f"[{cat_key}] concatenated_from key '{src_key}' is a special role "
                f"(full_access / consolidated) and cannot be used in concatenations."
            )
        resolved[src_key] = {
            "table":      src_settings["table"],
            "column":     src_settings["column"],
            "is_numeric": src_settings.get("is_numeric", False),
        }
    return resolved


def preprocess_concatenated_roles(df: DataFrame, config: dict) -> tuple:
    """
    Pre-processes the RLS dataframe and config for concatenated roles.
    Returns (enriched_df, enriched_config), ready for every function that follows.
    """
    import copy
    config = copy.deepcopy(config)

    concatenated = {}
    for key, settings in config.items():
        if "concatenated_from" not in settings:
            continue
        if settings.get("special"):
            print(f"   ⚠️  [{key}] has both 'concatenated_from' and 'special' — skipping.")
            continue
        try:
            resolved = _resolve_concatenated_sources(key, settings, config)
            concatenated[key] = resolved
        except ValueError as e:
            print(f"   ❌ {e}")

    if not concatenated:
        print("ℹ️  No concatenated roles defined — nothing to preprocess.")
        return df, config

    print(f"🔗 {len(concatenated)} concatenated role definition(s):\n")
    for cat_key, sources in concatenated.items():
        print(f"   · {cat_key}: {list(sources.keys())}")

    all_src_keys = set()
    for sources in concatenated.values():
        all_src_keys.update(sources.keys())

    src_key_to_col = {src_key: src_key for src_key in all_src_keys}

    combination_lookup = {
        frozenset(sources.keys()): cat_key
        for cat_key, sources in concatenated.items()
    }

    print(f"\n🔍 Looking for unmatched combinations in the data...")
    src_cols_in_df = {
        src_key: col_name
        for src_key, col_name in src_key_to_col.items()
        if col_name in df.columns
    }
    select_exprs = [
        F.when(F.col(col_name).isNotNull() & (F.col(col_name) != "") & (F.col(col_name) != "None"),
               F.lit(src_key)
        ).otherwise(F.lit(None)).alias(f"_src_{src_key}")
        for src_key, col_name in src_cols_in_df.items()
    ]
    df_check = df.select(select_exprs).distinct().collect()

    unmatched_combinations = set()
    for row in df_check:
        filled = frozenset(
            src_key for src_key in src_cols_in_df
            if row[f"_src_{src_key}"] is not None
        )
        if len(filled) < 2:
            continue
        if filled not in combination_lookup:
            unmatched_combinations.add(filled)

    if unmatched_combinations:
        print(f"\n   ⚠️  {len(unmatched_combinations)} unmatched combination(s).")
        print(f"   These users will fall back to INDIVIDUAL roles — check if that's intended:\n")
        for combo in sorted(unmatched_combinations, key=lambda s: sorted(s)):
            print(f"      · {sorted(combo)}")
        print()
    else:
        print(f"   ✅ All combinations are covered.\n")

    for src_key in all_src_keys:
        df = df.withColumn(f"_consumed_{src_key}", F.lit(False))

    print(f"🏗️  Building concatenated columns...\n")
    for cat_key, sources in concatenated.items():
        src_keys  = list(sources.keys())

        missing = [sk for sk in src_keys if sk not in df.columns]
        if missing:
            print(f"   ⚠️  [{cat_key}] Skipping — column(s) not found: {missing}")
            continue

        match_condition = F.lit(True)
        for src_key in src_keys:
            match_condition = match_condition & (
                F.col(src_key).isNotNull() &
                (F.col(src_key) != "") &
                (F.col(src_key) != "None")
            )

        sorted_src_keys = sorted(src_keys)
        concat_expr = F.col(sorted_src_keys[0])
        for sk in sorted_src_keys[1:]:
            concat_expr = F.concat(concat_expr, F.lit("_"), F.col(sk))

        synthetic_col = f"_cat_{cat_key}"
        df = df.withColumn(
            synthetic_col,
            F.when(match_condition, concat_expr).otherwise(F.lit(None))
        )

        for src_key in src_keys:
            df = df.withColumn(
                f"_consumed_{src_key}",
                F.col(f"_consumed_{src_key}") | match_condition
            )

        n_matched = df.where(F.col(synthetic_col).isNotNull()).count()
        print(f"   ✅ [{cat_key}] → {n_matched} row(s) | synthetic column: '{synthetic_col}'")

        filters_by_table = {}
        for src_key, src_def in sources.items():
            table      = src_def["table"]
            column     = src_def["column"]
            is_numeric = src_def.get("is_numeric", False)
            filters_by_table.setdefault(table, []).append(
                {"src_key": src_key, "column": column, "is_numeric": is_numeric}
            )

        config[cat_key]["_resolved_sources"] = sources
        config[cat_key]["_filters_by_table"] = filters_by_table
        config[cat_key]["column"]            = synthetic_col
        config[cat_key]["table"]             = None
        config[cat_key]["_is_concatenated"]  = True

    print(f"\n🧹 Nulling out consumed individual columns...")
    for src_key in all_src_keys:
        consumed_col   = f"_consumed_{src_key}"
        df_col_to_null = src_key
        if consumed_col not in df.columns or df_col_to_null not in df.columns:
            continue
        df = df.withColumn(
            df_col_to_null,
            F.when(F.col(consumed_col), F.lit(None)).otherwise(F.col(df_col_to_null))
        )
        n_nulled = df.where(F.col(consumed_col)).count()
        print(f"   · '{df_col_to_null}' nulled in {n_nulled} row(s)")

    print(f"\n🔤 Renaming synthetic columns...")
    for cat_key in concatenated:
        synthetic_col = f"_cat_{cat_key}"
        if synthetic_col in df.columns:
            df = df.withColumnRenamed(synthetic_col, cat_key)
            config[cat_key]["column"] = cat_key
            print(f"   · '{synthetic_col}' → '{cat_key}'")

    for src_key in all_src_keys:
        consumed_col = f"_consumed_{src_key}"
        if consumed_col in df.columns:
            df = df.drop(consumed_col)

    print(f"\n✅ Preprocessing complete.\n")
    return df, config

#### Create or Replace roles

In [ ]:
def create_or_replace_roles(config, dataset, workspace, global_filters, rls, config_keys=None):
    selected = {k: v for k, v in config.items() if config_keys is None or k in config_keys}
    print(f"⚫ Processing {len(selected)} config(s): {list(selected.keys())}\n")

    with connect_semantic_model(dataset=dataset, readonly=False, workspace=workspace) as tom:

        for config_key, settings in selected.items():
            special         = settings.get("special")
            is_concatenated = settings.get("_is_concatenated", False)

            # ── FULL ACCESS ───────────────────────────────────────────────────
            if special == "full_access":
                role_name = settings["prefix"]
                existing  = tom.model.Roles.Find(role_name)
                if existing is not None:
                    print(f"🟰  Role already existed: {role_name}")
                else:
                    print(f"✅ Creating: {role_name}")
                    tom.add_role(role_name=role_name)
                    print(f"   🔓 No DAX filter — full access role")
                continue

            # ── CONSOLIDATED / CONSOLIDATED_DEFAULT ─────────────────────────────
            # Both create the role the same way (same fixed_filter/table) — they
            # only differ in how membership is computed (see add_members_to_roles
            # and update_members_delta). The filter is refreshed even if the role
            # already exists, so config changes are picked up on re-run.
            if special in ("consolidated", "consolidated_default"):
                role_name = settings["prefix"]
                existing  = tom.model.Roles.Find(role_name)
                if existing is None:
                    print(f"✅ Creating: {role_name}")
                    tom.add_role(role_name=role_name)
                else:
                    print(f"🔄 Updating filter: {role_name}")
                tom.set_rls(
                    role_name=role_name,
                    table_name=settings["table"],
                    filter_expression=settings["fixed_filter"]
                )
                print(f"   🔒 {settings['table']}: {settings['fixed_filter']}")
                continue

            # ── CONCATENATED: one role per unique combination ───────────────────
            # The filter is refreshed even if the role already exists — this is
            # required so that fixes to the DAX-generation logic (e.g. the
            # is_numeric handling) actually reach roles created by earlier runs,
            # instead of being silently skipped forever.
            if is_concatenated:
                prefix           = settings["prefix"]
                synthetic_col    = settings["column"]
                filters_by_table = settings["_filters_by_table"]
                resolved_sources = settings["_resolved_sources"]

                if synthetic_col not in rls.columns:
                    print(f"   ⚠️  [{config_key}] Synthetic column '{synthetic_col}' not in RLS df — skipping.")
                    continue

                unique_values = (
                    rls.select(synthetic_col)
                    .where(F.col(synthetic_col).isNotNull())
                    .distinct()
                    .collect()
                )

                for row in unique_values:
                    combined_value = str(row[synthetic_col])
                    role_name      = f"{prefix}_{sanitize_role_name(combined_value)}"
                    existing       = tom.model.Roles.Find(role_name)

                    sorted_src_keys = sorted(resolved_sources.keys())
                    src_values      = combined_value.split("_", len(sorted_src_keys) - 1)

                    if len(src_values) != len(sorted_src_keys):
                        print(f"   ⚠️  Could not split '{combined_value}' into {len(sorted_src_keys)} part(s) — skipping.")
                        continue

                    src_value_map = dict(zip(sorted_src_keys, src_values))

                    # Guard: skip this role entirely if any required source value
                    # is blank (e.g. stored as "" rather than a real value) —
                    # producing "[Column] = " would be invalid/meaningless DAX.
                    blank_keys = [sk for sk, v in src_value_map.items() if not v]
                    if blank_keys:
                        print(f"   ⚠️  [{role_name}] Blank value for {blank_keys} — skipping this role.")
                        continue

                    dax_calls = []
                    for table, col_defs in filters_by_table.items():
                        exprs = []
                        for cd in col_defs:
                            val = src_value_map[cd["src_key"]]
                            if cd.get("is_numeric"):
                                # Numeric column (e.g. DocType Integer) — no quotes,
                                # otherwise DAX raises a type-comparison error.
                                exprs.append(f'[{cd["column"]}] = {val}')
                            else:
                                exprs.append(f'[{cd["column"]}] = "{val}"')
                        dax_expr = " && ".join(f"({e})" for e in exprs)
                        dax_calls.append((table, dax_expr))

                    for gf in global_filters:
                        gf_expr = f'[{gf["column"]}] = "{gf["value"]}"'
                        merged = False
                        for i, (tbl, expr) in enumerate(dax_calls):
                            if tbl == gf["table"]:
                                dax_calls[i] = (tbl, f"({expr}) && ({gf_expr})")
                                merged = True
                                break
                        if not merged:
                            dax_calls.append((gf["table"], gf_expr))

                    if existing is None:
                        print(f"✅ Creating: {role_name}")
                        tom.add_role(role_name=role_name)
                    else:
                        print(f"🔄 Updating filter: {role_name}")
                    for table, dax_expr in dax_calls:
                        tom.set_rls(role_name=role_name, table_name=table, filter_expression=dax_expr)
                        print(f"   🔒 {table}: {dax_expr}")

                continue

            # ── STANDARD: one role per unique value ─────────────────────────────
            # The filter is refreshed even if the role already exists.
            prefix     = settings["prefix"]
            source_col = settings.get("source_column", config_key)

            if source_col not in rls.columns:
                print(f"   ⚠️  [{config_key}] Column '{source_col}' not in RLS df — skipping.")
                continue

            unique_values = (
                rls.select(source_col)
                .where(F.col(source_col).isNotNull())
                .distinct()
                .collect()
            )
            for row in unique_values:
                value     = str(row[source_col])
                role_name = f"{prefix}_{sanitize_role_name(value)}"
                filters   = build_dax_filters(settings, value, global_filters)

                existing = tom.model.Roles.Find(role_name)
                if existing is None:
                    print(f"✅ Creating: {role_name}")
                    tom.add_role(role_name=role_name)
                else:
                    print(f"🔄 Updating filter: {role_name}")
                for table, dax_filter in filters:
                    tom.set_rls(role_name=role_name, table_name=table, filter_expression=dax_filter)
                    print(f"   🔒 {table}: {dax_filter}")

        tom.model.SaveChanges()
        print("\n✅ All roles saved.")

#### Add members to roles

In [ ]:
def add_members_to_roles(
    config, dataset, workspace, rls,
    username_col="Username",
    config_keys=None,
    chunk_size=50
):
    import math
    selected = {k: v for k, v in config.items() if config_keys is None or k in config_keys}

    if config_keys:
        print(f"⚫ Processing {len(selected)}/{len(config)} config(s): {list(selected.keys())}\n")
    else:
        print(f"⚫ Processing all {len(selected)} config(s)\n")

    # ── Step 1: Valid UPNs from Spark ─────────────────────────────────────────
    all_upns = (
        rls.select(username_col)
        .where(F.col(username_col).isNotNull())
        .distinct()
        .collect()
    )
    valid_upns = {
        str(row[username_col]).strip()
        for row in all_upns
        if "@" in str(row[username_col]) and " " not in str(row[username_col]).strip()
    }
    print(f"📊 {len(valid_upns)} valid UPN(s) found\n")

    # ── Step 2: Build members_map from Spark ──────────────────────────────────
    members_map    = {}
    failed_members = []

    for config_key, settings in selected.items():
        prefix  = settings["prefix"]
        special = settings.get("special")

        # ── FULL ACCESS ───────────────────────────────────────────────────────
        if special == "full_access":
            role_name  = f"{prefix}"
            source_col = settings["source_column"]
            source_val = settings["source_value"]

            matched = (
                rls.where(F.col(source_col) == source_val)
                .select(username_col)
                .where(F.col(username_col).isNotNull())
                .distinct()
                .collect()
            )
            added = 0
            for r in matched:
                upn = str(r[username_col]).strip()
                if upn not in valid_upns:
                    failed_members.append({"role": role_name, "username": upn, "reason": "Invalid UPN format"})
                    continue
                members_map.setdefault(role_name, []).append(upn)
                added += 1

            print(f"   🔓 FullAccess role: {added} members")
            continue

        # ── CONSOLIDATED (opt-in: flag + every other column empty) ─────────────
        if special == "consolidated":
            role_name  = f"{prefix}"
            source_col = settings["source_column"]
            source_val = settings["source_value"]

            matched = (
                rls.where(F.col(source_col) == source_val)
                .select(username_col)
                .where(F.col(username_col).isNotNull())
                .distinct()
                .collect()
            )
            added = 0
            for r in matched:
                upn = str(r[username_col]).strip()
                if upn not in valid_upns:
                    failed_members.append({"role": role_name, "username": upn, "reason": "Invalid UPN format"})
                    continue
                members_map.setdefault(role_name, []).append(upn)
                added += 1

            print(f"   🔒 Consolidated role: {added} members (Is_Consolidated == '{source_val}')")
            continue

        # ── CONSOLIDATED_DEFAULT: everyone EXCEPT those eligible for ────────────
        # the keys listed in exclude_special_keys (e.g. FullAccess).
        if special == "consolidated_default":
            role_name    = f"{prefix}"
            exclude_keys = settings.get("exclude_special_keys", [])

            exclude_filter = F.lit(False)
            for exkey in exclude_keys:
                ex_settings = config[exkey]
                ex_col = ex_settings["source_column"]
                ex_val = ex_settings["source_value"]
                exclude_filter = exclude_filter | (F.col(ex_col) == ex_val)

            eligible = rls.where(~exclude_filter) if exclude_keys else rls

            matched = (
                eligible.select(username_col)
                .where(F.col(username_col).isNotNull())
                .distinct()
                .collect()
            )
            added = 0
            for r in matched:
                upn = str(r[username_col]).strip()
                if upn not in valid_upns:
                    failed_members.append({"role": role_name, "username": upn, "reason": "Invalid UPN format"})
                    continue
                members_map.setdefault(role_name, []).append(upn)
                added += 1

            excl_note = f" (excluding: {exclude_keys})" if exclude_keys else ""
            print(f"   🔒 Consolidated (default-all) role: {added} members{excl_note}")
            continue

        # ── STANDARD: one role per unique value ─────────────────────────────────
        source_col = settings.get("source_column", config_key)

        if source_col not in rls.columns:
            print(f"   ⚠️ Skipping {config_key} — column '{source_col}' not in RLS df")
            continue

        unique_values = (
            rls.select(source_col)
            .where(F.col(source_col).isNotNull())
            .distinct()
            .collect()
        )
        for row in unique_values:
            value     = str(row[source_col])
            role_name = f"{prefix}_{sanitize_role_name(value)}"

            raw_members = (
                rls.where(F.col(source_col) == value)
                .select(username_col)
                .where(F.col(username_col).isNotNull())
                .distinct()
                .collect()
            )
            for r in raw_members:
                upn = str(r[username_col]).strip()
                if upn not in valid_upns:
                    failed_members.append({"role": role_name, "username": upn, "reason": "Invalid UPN format"})
                    continue
                members_map.setdefault(role_name, []).append(upn)

    total_members = sum(len(v) for v in members_map.values())
    total_chunks  = math.ceil(total_members / chunk_size) if total_members else 0
    print(f"\n📋 {total_members} membership(s) → {total_chunks} chunk(s) of {chunk_size}\n")

    # ── Step 3: Flatten to (role_name, upn) pairs ─────────────────────────────
    all_pairs = [
        (role_name, upn)
        for role_name, upns in members_map.items()
        for upn in upns
    ]

    # ── Step 4: Save in batches, with bisection on failure ────────────────────
    def process_chunk(pairs, chunk_index):
        chunk_failed = []
        with connect_semantic_model(dataset=dataset, readonly=False, workspace=workspace) as tom:
            for role_name, upn in pairs:
                if tom.model.Roles.Find(role_name) is None:
                    chunk_failed.append({"role": role_name, "username": upn, "reason": "Role not found"})
                    continue
                try:
                    tom.add_role_member(role_name=role_name, member=upn, role_member_type="User")
                except Exception as e:
                    chunk_failed.append({"role": role_name, "username": upn, "reason": str(e)[:200]})

            try:
                tom.model.SaveChanges()
                print(f"   ✅ Chunk {chunk_index}: {len(pairs)} members saved")
            except Exception as e:
                print(f"   ⚠️ Chunk {chunk_index} failed ({str(e)[:80]}) — bisecting...")
                if len(pairs) == 1:
                    role_name, upn = pairs[0]
                    print(f"   ❌ Isolated invalid member: {upn} → {role_name}")
                    chunk_failed.append({"role": role_name, "username": upn, "reason": str(e)[:200]})
                else:
                    mid = len(pairs) // 2
                    chunk_failed += process_chunk(pairs[:mid], f"{chunk_index}a")
                    chunk_failed += process_chunk(pairs[mid:], f"{chunk_index}b")

        return chunk_failed

    # ── Step 5: Run every chunk ─────────────────────────────────────────────────
    chunks = [all_pairs[i:i + chunk_size] for i in range(0, len(all_pairs), chunk_size)]
    for i, chunk in enumerate(chunks, 1):
        print(f"🔄 Chunk {i}/{len(chunks)} ({len(chunk)} members)...")
        failed_members.extend(process_chunk(chunk, i))

    # ── Step 6: Final report ──────────────────────────────────────────────────
    success_count = total_members - len(failed_members)
    print(f"\n📊 Results: {success_count} added successfully, {len(failed_members)} failed")

    if failed_members:
        print(f"\n❌ {len(failed_members)} failure(s):")
        print(json.dumps(failed_members, indent=2))
        failed_df = spark.createDataFrame(failed_members)
        (
            failed_df.coalesce(1)
            .write.mode("overwrite")
            .json("Files/rls_failed_members")
        )
        print("💾 Saved to: Files/rls_failed_members/")
    else:
        print("🎉 All members added successfully — no failures!")

    return failed_members

#### Drop unused roles

In [ ]:
def drop_unused_roles(config, dataset, workspace, rls, config_keys=None):
    selected = {k: v for k, v in config.items() if config_keys is None or k in config_keys}

    if config_keys:
        print(f"⚫ Processing {len(selected)}/{len(config)} config(s): {list(selected.keys())}\n")
    else:
        print(f"⚫ Processing all {len(selected)} config(s)\n")

    expected_roles = set()

    for config_key, settings in selected.items():
        prefix  = settings["prefix"]
        special = settings.get("special")

        # ── Special roles: fixed name, no value suffix ──────────────────────────
        if special in ("full_access", "consolidated", "consolidated_default"):
            expected_roles.add(f"{prefix}")
            print(f"   📌 Special role kept: {prefix}")
            continue

        # ── Standard roles ────────────────────────────────────────────────────
        source_col = settings.get("source_column", config_key)

        if source_col not in rls.columns:
            print(f"   ⚠️ Skipping {config_key} — column '{source_col}' not in RLS df")
            continue

        current_values = (
            rls.select(source_col)
            .where(F.col(source_col).isNotNull())
            .distinct()
            .collect()
        )
        for row in current_values:
            value = str(row[source_col])
            expected_roles.add(f"{prefix}_{sanitize_role_name(value)}")

    print(f"\n📋 {len(expected_roles)} role(s) expected from current RLS data\n")

    dropped = []
    kept    = []

    with connect_semantic_model(dataset=dataset, readonly=False, workspace=workspace) as tom:
        for config_key, settings in selected.items():
            prefix  = settings["prefix"]
            special = settings.get("special")

            owned_roles = [
                r for r in tom.model.Roles
                if r.Name.startswith(f"{prefix}_") or
                   (special and r.Name == prefix)
            ]

            print(f"📂 {config_key} — {len(owned_roles)} role(s) with prefix '{prefix}'")

            for role in owned_roles:
                if role.Name not in expected_roles:
                    print(f"  🗑️  Dropping:  {role.Name}")
                    tom.model.Roles.Remove(role)
                    dropped.append(role.Name)
                else:
                    print(f"  ✅ Keeping:   {role.Name}")
                    kept.append(role.Name)

        if dropped:
            tom.model.SaveChanges()
            print(f"\n✅ Dropped {len(dropped)} unused role(s).")
        else:
            print("\n✅ No unused roles found — nothing to drop.")

    print(f"\n📊 Summary: {len(kept)} kept, {len(dropped)} dropped")
    if dropped:
        print("\nDropped roles:")
        for r in dropped:
            print(f"  - {r}")

    return dropped

#### Remove all members from roles

In [ ]:
# ── STEP 3: Remove all members from roles ─────────────────────────────────────
def remove_all_members_from_roles(config, dataset, workspace, config_keys=None):
    """
    Removes all members from the managed roles, without dropping the roles.
    config_keys: list of keys to process. If None, all are processed.
    """
    selected = {k: v for k, v in config.items() if config_keys is None or k in config_keys}

    print(f"⚫ Processing {len(selected)} config(s): {list(selected.keys())}\n")

    with connect_semantic_model(dataset=dataset, readonly=False, workspace=workspace) as tom:
        removed_total = 0

        for config_key, settings in selected.items():
            prefix = settings["prefix"]

            for role in list(tom.model.Roles):
                if not role.Name.startswith(f"{prefix}_") and role.Name != prefix:
                    continue

                members = list(role.Members)
                if not members:
                    print(f"   ℹ️ No members: {role.Name}")
                    continue

                for member in members:
                    role.Members.Remove(member)
                    print(f"   🗑️ Removed: {member.MemberName} ← {role.Name}")
                    removed_total += 1

        tom.model.SaveChanges()
        print(f"\n✅ Done. {removed_total} member(s) removed.")

#### Sync members to roles

In [ ]:
# ── STEP 4: Sync members (clear + re-apply from rls table) ────────────────────
def sync_members_to_roles(config, dataset, workspace, rls, username_col="Username", config_keys=None):
    """
    Full sync: removes all current members and re-adds them from the rls
    table. Use after large-scale changes to get a clean state. Expensive on
    roles with many members — prefer update_members_delta for daily maintenance.
    """
    print("🔁 Step 1/2 — Clearing existing members...\n")
    remove_all_members_from_roles(
        config=config,
        dataset=dataset,
        workspace=workspace,
        config_keys=config_keys
    )

    print("\n🔁 Step 2/2 — Re-adding members from rls table...\n")
    failed = add_members_to_roles(
        config=config,
        dataset=dataset,
        workspace=workspace,
        rls=rls,
        username_col=username_col,
        config_keys=config_keys
    )

    return failed

#### Update members delta

In [ ]:
def update_members_delta(
    config, dataset, workspace, rls,
    username_col="Username",
    config_keys=None,
    chunk_size=1,          # defaults to the original one-by-one behavior (no real batching)
    skip_members=None      # list of (role_name, upn_compare) tuples to exclude
):
    """
    Incremental (idempotent) sync based on the data delta.

    Flow:
      0. Cleans dirty members off the server before starting
      1. Reads current members from the semantic model → df_current
      2. Builds the target state from the RLS table → df_target
      3. Computes the delta in Spark using upn_compare (anti-joins)
      4. Applies changes IN BATCHES (chunk_size) with bisection on failure
         — chunk_size=1 replicates the original behavior (one at a time);
           raising it drastically cuts the number of SaveChanges calls needed.
    """
    import time, re, math

    selected = {k: v for k, v in config.items() if config_keys is None or k in config_keys}
    if config_keys:
        print(f"⚫ Processing {len(selected)}/{len(config)} config(s): {list(selected.keys())}\n")
    else:
        print(f"⚫ Processing all {len(selected)} config(s)\n")

    # ── UPN normalization ─────────────────────────────────────────────────────
    def normalize_upn(upn: str) -> str:
        upn = upn.strip().lower()
        for suffix in ["#azuread", "#externalaad"]:
            if upn.endswith(suffix):
                upn = upn[: -len(suffix)]
        return upn.strip()

    # ── UPN validation ────────────────────────────────────────────────────────
    def is_valid_upn(upn: str):
        upn = normalize_upn(upn)
        if upn.count("@") != 1:
            return False, "Multiple or missing @"
        if " " in upn:
            return False, "Contains spaces"
        local, domain = upn.split("@")
        if not local:
            return False, "Empty local part"
        if "." not in domain:
            return False, "Invalid domain"
        if "#ext#" in upn:
            if not re.match(r'^.+#ext#@.+\.onmicrosoft\.com$', upn):
                return False, "Malformed #EXT# UPN"
        return True, "OK"

    # ── Step 0: Clean dirty members off the server ────────────────────────────
    print("🧹 Scanning for invalid members on the server...")
    dirty = []
    with connect_semantic_model(dataset=dataset, workspace=workspace, readonly=True) as tom:
        for role in tom.model.Roles:
            for member in role.Members:
                raw = member.MemberName
                ok, reason = is_valid_upn(raw)
                if not ok:
                    print(f"   ❌ DIRTY: '{raw}' in '{role.Name}' — {reason}")
                    dirty.append((role.Name, raw))

    if dirty:
        print(f"\n   ⚠️  {len(dirty)} dirty member(s) found — cleaning up...")
        with connect_semantic_model(dataset=dataset, workspace=workspace, readonly=False) as tom:
            for role_name, member_name in dirty:
                role_obj = tom.model.Roles.Find(role_name)
                if role_obj:
                    member_obj = next(
                        (m for m in role_obj.Members if m.MemberName == member_name), None
                    )
                    if member_obj:
                        role_obj.Members.Remove(member_obj)
                        print(f"   🗑️  Removed: '{member_name}' from '{role_name}'")
            tom.model.SaveChanges()
            print(f"   ✅ {len(dirty)} dirty member(s) removed\n")
    else:
        print("   ✅ No dirty members found\n")

    # ── Step 1: Read current members from the model → df_current ─────────────
    print("🔍 Reading current members from the semantic model...")
    with connect_semantic_model(dataset=dataset, workspace=workspace, readonly=True) as tom:
        current_rows = []
        for role in tom.model.Roles:
            for member in role.Members:
                current_rows.append({
                    "role_name":   role.Name,
                    "upn_compare": normalize_upn(member.MemberName),
                    "upn_raw":     member.MemberName.strip()
                })

    df_current = spark.createDataFrame(current_rows) if current_rows else spark.createDataFrame(
        [], schema="role_name STRING, upn_compare STRING, upn_raw STRING"
    )
    print(f"   📋 {df_current.count()} current membership(s) in the model\n")

    # ── Step 2: Resolve UPNs ──────────────────────────────────────────────────
    print("🔄 Resolving UPNs (compare=model_upn, add=Username)...")
    has_model_upn = "model_upn" in rls.columns
    if has_model_upn:
        rls = rls.withColumn(
            "_upn_compare",
            F.when(
                F.col("model_upn").isNotNull() &
                (F.col("model_upn") != "") &
                (F.col("model_upn") != "None"),
                F.lower(F.col("model_upn"))
            ).otherwise(F.lower(F.col(username_col)))
        ).withColumn(
            "_upn_add",
            F.lower(F.col(username_col))
        )
        print(f"   ✅ model_upn found — compare=model_upn, add=Username\n")
    else:
        rls = rls.withColumn("_upn_compare", F.lower(F.col(username_col))) \
                 .withColumn("_upn_add",     F.lower(F.col(username_col)))
        print(f"   ℹ️ No model_upn — compare=add=Username\n")

    # ── Step 3: Build target state from RLS → df_target ───────────────────────
    print("🏗️  Building target state from the RLS table...")

    valid_upns_df = (
        rls.select(F.col("_upn_compare").alias("upn_compare"))
        .where(F.col("upn_compare").isNotNull())
        .where(F.col("upn_compare").contains("@"))
        .where(~F.col("upn_compare").contains(" "))
        .distinct()
    )

    sanitize_udf = F.udf(sanitize_role_name)   # reuses the same function used to create roles

    target_rows    = []
    failed_members = []

    for config_key, settings in selected.items():
        prefix  = settings["prefix"]
        special = settings.get("special")

        # ── FULL ACCESS ───────────────────────────────────────────────────────
        if special == "full_access":
            role_name  = prefix
            source_col = settings["source_column"]
            source_val = settings["source_value"]

            raw = (
                rls.where(F.col(source_col) == source_val)
                .select(
                    F.col("_upn_compare").alias("upn_compare"),
                    F.col("_upn_add").alias("upn_add")
                )
                .where(F.col("upn_compare").isNotNull())
                .join(valid_upns_df, "upn_compare")
                .withColumn("role_name", F.lit(role_name))
                .select("role_name", "upn_compare", "upn_add")
                .distinct()
                .collect()
            )
            valid_rows = []
            for r in raw:
                ok, reason = is_valid_upn(r.upn_compare)
                if ok:
                    valid_rows.append(r)
                else:
                    print(f"   ⚠️ Skipped ({reason}): {r.upn_compare}")
                    failed_members.append({"role": role_name, "username": r.upn_add, "reason": reason})

            if valid_rows:
                target_rows.append(spark.createDataFrame(valid_rows))
            print(f"   🔓 FullAccess '{role_name}': {len(valid_rows)} target")
            continue

        # ── CONSOLIDATED (opt-in: flag + every other column empty) ─────────────
        if special == "consolidated":
            role_name   = prefix
            source_col  = settings["source_column"]
            source_val  = settings["source_value"]
            base_ignore = ["PK", "Username", "FullAccess", "Is_Consolidated", "model_upn",
                           "_upn_compare", "_upn_add"]
            ignore_cols = list(set(settings.get("ignore_cols", base_ignore) + base_ignore))
            filter_cols = [c for c in rls.columns if c not in ignore_cols]

            consolidated_filter = F.col(source_col) == source_val
            for col in filter_cols:
                consolidated_filter = consolidated_filter & (
                    F.col(col).isNull() | (F.col(col) == "") | (F.col(col) == "None")
                )

            raw = (
                rls.where(consolidated_filter)
                .select(
                    F.col("_upn_compare").alias("upn_compare"),
                    F.col("_upn_add").alias("upn_add")
                )
                .where(F.col("upn_compare").isNotNull())
                .join(valid_upns_df, "upn_compare")
                .withColumn("role_name", F.lit(role_name))
                .select("role_name", "upn_compare", "upn_add")
                .distinct()
                .collect()
            )
            valid_rows = []
            for r in raw:
                ok, reason = is_valid_upn(r.upn_compare)
                if ok:
                    valid_rows.append(r)
                else:
                    print(f"   ⚠️ Skipped ({reason}): {r.upn_compare}")
                    failed_members.append({"role": role_name, "username": r.upn_add, "reason": reason})

            if valid_rows:
                target_rows.append(spark.createDataFrame(valid_rows))
            print(f"   🔒 Consolidated '{role_name}': {len(valid_rows)} target (only-consolidated users)")
            continue

        # ── CONSOLIDATED_DEFAULT: everyone EXCEPT eligible for exclude_special_keys ──
        if special == "consolidated_default":
            role_name    = prefix
            exclude_keys = settings.get("exclude_special_keys", [])

            exclude_filter = F.lit(False)
            for exkey in exclude_keys:
                ex_settings = config[exkey]
                ex_col = ex_settings["source_column"]
                ex_val = ex_settings["source_value"]
                exclude_filter = exclude_filter | (F.col(ex_col) == ex_val)

            eligible = rls.where(~exclude_filter) if exclude_keys else rls

            raw = (
                eligible.select(
                    F.col("_upn_compare").alias("upn_compare"),
                    F.col("_upn_add").alias("upn_add")
                )
                .where(F.col("upn_compare").isNotNull())
                .join(valid_upns_df, "upn_compare")
                .withColumn("role_name", F.lit(role_name))
                .select("role_name", "upn_compare", "upn_add")
                .distinct()
                .collect()
            )
            valid_rows = []
            for r in raw:
                ok, reason = is_valid_upn(r.upn_compare)
                if ok:
                    valid_rows.append(r)
                else:
                    print(f"   ⚠️ Skipped ({reason}): {r.upn_compare}")
                    failed_members.append({"role": role_name, "username": r.upn_add, "reason": reason})

            if valid_rows:
                target_rows.append(spark.createDataFrame(valid_rows))

            excl_note = f" (excl: {exclude_keys})" if exclude_keys else ""
            print(f"   🔒 Consolidated_default '{role_name}': {len(valid_rows)} target{excl_note}")
            continue

        # ── STANDARD ─────────────────────────────────────────────────────────
        source_col = settings.get("source_column", config_key)
        if source_col not in rls.columns:
            print(f"   ⚠️ Skipping {config_key} — column '{source_col}' not in RLS df")
            continue

        raw = (
            rls.where(F.col(source_col).isNotNull())
            .select(
                F.concat(F.lit(prefix + "_"), sanitize_udf(F.col(source_col))).alias("role_name"),
                F.col("_upn_compare").alias("upn_compare"),
                F.col("_upn_add").alias("upn_add")
            )
            .where(F.col("upn_compare").isNotNull())
            .join(valid_upns_df, "upn_compare")
            .select("role_name", "upn_compare", "upn_add")
            .distinct()
            .collect()
        )
        valid_rows = []
        for r in raw:
            ok, reason = is_valid_upn(r.upn_compare)
            if ok:
                valid_rows.append(r)
            else:
                print(f"   ⚠️ Skipped ({reason}): {r.upn_compare} → {r.role_name}")
                failed_members.append({"role": r.role_name, "username": r.upn_add, "reason": reason})

        if valid_rows:
            target_rows.append(spark.createDataFrame(valid_rows))

    # ── Merge into df_target ──────────────────────────────────────────────────
    if target_rows:
        df_target = target_rows[0]
        for df_part in target_rows[1:]:
            df_target = df_target.union(df_part)
        df_target = df_target.distinct()
    else:
        df_target = spark.createDataFrame([], schema="role_name STRING, upn_compare STRING, upn_add STRING")

    # ── Filter df_current to managed roles only ───────────────────────────────
    managed_prefixes   = [settings["prefix"] for settings in selected.values()]
    current_role_names = [row.role_name for row in df_current.select("role_name").distinct().collect()]
    managed_roles      = [
        r for r in current_role_names
        if any(r == p or r.startswith(p + "_") for p in managed_prefixes)
    ]
    df_current_filtered = df_current.where(F.col("role_name").isin(managed_roles)) if managed_roles else \
        spark.createDataFrame([], schema="role_name STRING, upn_compare STRING, upn_raw STRING")

    target_count  = df_target.count()
    current_count = df_current_filtered.count()
    print(f"\n📊 Current state (managed roles): {current_count} membership(s)")
    print(f"📊 Target state:                   {target_count} membership(s)")

    # ── Step 4: Delta based on upn_compare ────────────────────────────────────
    df_to_add = df_target.join(
        df_current_filtered.select("role_name", "upn_compare"),
        ["role_name", "upn_compare"], "left_anti"
    )
    df_to_remove = df_current_filtered.join(
        df_target.select("role_name", "upn_compare"),
        ["role_name", "upn_compare"], "left_anti"
    )

    # ── Step 4b: Exclude pairs blocked by skip_members ────────────────────────
    if skip_members:
        skip_df = spark.createDataFrame(
            [(r.lower(), u.lower()) for r, u in skip_members],
            schema="role_name STRING, upn_compare STRING"
        )
        before_skip = df_to_add.count()
        df_to_add = df_to_add.join(skip_df, ["role_name", "upn_compare"], "left_anti")
        after_skip = df_to_add.count()
        skipped_n  = before_skip - after_skip
        if skipped_n > 0:
            print(f"\n   ⏭️  {skipped_n} pair(s) excluded by skip_members:")
            for r, u in skip_members:
                print(f"      · {u} → {r}")

    n_add    = df_to_add.count()
    n_remove = df_to_remove.count()

    print(f"\n📊 Computed delta:")
    print(f"   ➕ To add:    {n_add} membership(s)")
    print(f"   ➖ To remove: {n_remove} membership(s)")

    if n_add == 0 and n_remove == 0:
        print("\n✅ No changes — everything is already in sync.")
        return failed_members

    pairs_add    = [(r.role_name, r.upn_compare, r.upn_add) for r in df_to_add.collect()]
    pairs_remove = [(r.role_name, r.upn_compare, r.upn_raw) for r in df_to_remove.collect()]

    # ── Step 5: Apply changes IN BATCHES (chunk_size) with bisection ─────────
    all_failed = []

    def apply_chunk(pairs, operation, chunk_label):
        """
        Applies a batch of (role_name, upn_compare, upn_action) in a single
        connection + SaveChanges. On failure, bisects recursively to isolate
        the problematic member(s) without blocking the rest of the batch.
        """
        chunk_failed = []
        chunk_ok     = []

        with connect_semantic_model(dataset=dataset, readonly=False, workspace=workspace) as tom:
            for role_name, upn_compare, upn_action in pairs:
                role_obj = tom.model.Roles.Find(role_name)
                if role_obj is None:
                    chunk_failed.append((role_name, upn_compare, upn_action, "Role not found"))
                    continue
                try:
                    if operation == "add":
                        already = any(normalize_upn(m.MemberName) == upn_compare for m in role_obj.Members)
                        if already:
                            chunk_ok.append((role_name, upn_compare, upn_action, "already_member"))
                            continue
                        tom.add_role_member(role_name=role_name, member=upn_action, role_member_type="User")
                    else:
                        member_obj = next(
                            (m for m in role_obj.Members if normalize_upn(m.MemberName) == upn_compare),
                            None
                        )
                        if not member_obj:
                            chunk_ok.append((role_name, upn_compare, upn_action, "already_removed"))
                            continue
                        role_obj.Members.Remove(member_obj)
                    chunk_ok.append((role_name, upn_compare, upn_action, "pending_save"))
                except Exception as e:
                    chunk_failed.append((role_name, upn_compare, upn_action, str(e)[:200]))

            try:
                tom.model.SaveChanges()
                print(f"   ✅ Chunk {chunk_label}: {len(chunk_ok)} OK")
            except Exception as e:
                print(f"   ⚠️ Chunk {chunk_label} failed ({str(e)[:80]}) — bisecting...")
                if len(pairs) == 1:
                    role_name, upn_compare, upn_action = pairs[0]
                    return [], [(role_name, upn_compare, upn_action, str(e)[:200])]
                mid = len(pairs) // 2
                ok_a, fail_a = apply_chunk(pairs[:mid], operation, f"{chunk_label}a")
                ok_b, fail_b = apply_chunk(pairs[mid:], operation, f"{chunk_label}b")
                return ok_a + ok_b, fail_a + fail_b

        return chunk_ok, chunk_failed

    def run_operation(pairs, operation, label):
        results_ok, results_failed = [], []
        chunks = [pairs[i:i + chunk_size] for i in range(0, len(pairs), chunk_size)]
        for i, chunk in enumerate(chunks, 1):
            print(f"🔄 {label} batch {i}/{len(chunks)} ({len(chunk)} member(s))...")
            ok, failed = apply_chunk(chunk, operation, str(i))
            results_ok.extend(ok)
            results_failed.extend(failed)
        return results_ok, results_failed

    # ── ADD ───────────────────────────────────────────────────────────────────
    if pairs_add:
        print(f"\n➕ Adding {len(pairs_add)} member(s) (chunk_size={chunk_size})...")
        ok_add, failed_add = run_operation(pairs_add, "add", "ADD")
        for role_name, upn_compare, upn_add, reason in failed_add:
            all_failed.append({"operation": "add", "role": role_name, "username": upn_add,
                                "upn_compare": upn_compare, "reason": reason})
        print(f"   📊 ADD: {len(ok_add)} OK, {len(failed_add)} failed")

    # ── REMOVE ────────────────────────────────────────────────────────────────
    if pairs_remove:
        print(f"\n➖ Removing {len(pairs_remove)} member(s) (chunk_size={chunk_size})...")
        ok_remove, failed_remove = run_operation(pairs_remove, "remove", "REMOVE")
        for role_name, upn_compare, upn_raw, reason in failed_remove:
            all_failed.append({"operation": "remove", "role": role_name, "username": upn_raw,
                                "upn_compare": upn_compare, "reason": reason})
        print(f"   📊 REMOVE: {len(ok_remove)} OK, {len(failed_remove)} failed")

    # ── Final summary ─────────────────────────────────────────────────────────
    all_failed = failed_members + all_failed
    n_ok       = (n_add + n_remove) - len(all_failed)
    print(f"\n📊 Final result: {n_ok} operation(s) OK, {len(all_failed)} failed")

    if all_failed:
        print(f"\n❌ {len(all_failed)} failure(s):")
        print(json.dumps(all_failed, indent=2))
        try:
            failed_df = spark.createDataFrame(all_failed)
            failed_df.coalesce(1).write.mode("overwrite").json("Files/rls_failed_members_delta")
            print("💾 Saved to: Files/rls_failed_members_delta/")
        except Exception as write_err:
            print(f"⚠️ Could not save the log to OneLake: {write_err}")
    else:
        print("🎉 Delta applied with no errors!")

    return all_failed

#### Audit — snapshots, change log, run summary and alerts

Tables are written by ABFS OneLake path via `audit_path` (e.g. `path_control`
from `Connections`, ending in `/`) — reliable regardless of which lakehouse
is attached to the notebook. If `audit_path` is left as `None`, tables are
written relative to the notebook's default lakehouse.

Alerts are sent by email via Microsoft Graph, reusing `Email notifications`
(same pattern as `dq_AlertSending`): the token is requested once with
`get_graph_token()` and reused, and sending uses `send_email()` defined there.

⚠️ This notebook must be loaded AFTER `Email notifications`:
```
%run Email notifications
%run RLS_Management_Functions
```
so `TENANT_ID`, `CLIENT_ID`, `CLIENT_SECRET`, `SENDER_EMAIL`, `get_graph_token`,
`send_email` are already in scope when `send_alert` needs them.

In [ ]:
import uuid
from datetime import datetime

def _audit_table_path(table_name, audit_path=None):
    """
    Returns the write path for an audit table:
      - If audit_path is given (e.g. path_control from Connections) → explicit
        ABFS path, independent of which lakehouse is attached to the notebook.
      - If not → path relative to the notebook's default lakehouse.
    """
    if audit_path:
        return f"{audit_path}{table_name}"
    return f"Tables/{table_name}"


def _write_delta(df, table_name, mode="append", audit_path=None):
    path = _audit_table_path(table_name, audit_path)
    df.write.format("delta").mode(mode).save(path)


def send_alert(message, alert_to=None, subject="RLS Audit Alert", severity="warning"):
    """
    Sends an alert by email via Microsoft Graph, reusing get_graph_token()/
    send_email() from 'Email notifications' (requires a prior %run). If
    alert_to is None, only prints — useful in DEV.
    """
    icon = {"critical": "🔴", "warning": "🚨", "info": "ℹ️"}.get(severity, "🚨")
    full_message = f"{icon} [{severity.upper()}] {message}"
    print(full_message)

    if alert_to:
        try:
            graph_token = get_graph_token()   # defined in Email notifications
            send_email(
                subject=f"{icon} {subject}",
                html_body=f"<p>{full_message.replace(chr(10), '<br>')}</p>",
                to=alert_to,
                access_token=graph_token,
            )
            print(f"   📧 Alert sent by email → {alert_to}")
        except Exception as e:
            print(f"   ⚠️ Could not send the alert email: {e}")


def snapshot_current_membership(dataset, workspace, run_id, snapshot_type, audit_path=None):
    """
    Saves a full snapshot (role, user) of current membership in the model.
    snapshot_type: "before" or "after".
    Returns (count, dataframe).
    """
    current_ts = datetime.utcnow().isoformat()

    with connect_semantic_model(dataset=dataset, workspace=workspace, readonly=True) as tom:
        rows = [
            {
                "run_id": run_id,
                "run_timestamp": current_ts,
                "role_name": role.Name,
                "upn": member.MemberName,
                "snapshot_type": snapshot_type
            }
            for role in tom.model.Roles for member in role.Members
        ]

    df_snap = spark.createDataFrame(rows) if rows else spark.createDataFrame(
        [], schema="run_id STRING, run_timestamp STRING, role_name STRING, upn STRING, snapshot_type STRING"
    )

    if rows:
        _write_delta(df_snap, "rls_membership_snapshot", audit_path=audit_path)
    else:
        print("   ℹ️ No memberships to capture (empty model or roles with no members)")

    print(f"📸 Snapshot '{snapshot_type}' saved: {len(rows)} membership(s)")
    return len(rows), df_snap


def snapshot_run_summary(dataset, workspace, env, run_id, before_count, after_count, audit_path=None):
    from pyspark.sql.types import StructType, StructField, StringType, LongType, DoubleType

    current_ts = datetime.utcnow().isoformat()
    delta = after_count - before_count
    delta_pct = round(delta / before_count, 4) if before_count else None

    row = {
        "run_id":           run_id,
        "run_timestamp":    current_ts,
        "environment":      env,
        "model":            dataset,
        "workspace":        workspace,
        "members_before":   before_count,
        "members_after":    after_count,
        "delta":            delta,
        "delta_pct":        delta_pct,
    }

    schema = StructType([
        StructField("run_id",         StringType(), True),
        StructField("run_timestamp",  StringType(), True),
        StructField("environment",    StringType(), True),
        StructField("model",          StringType(), True),
        StructField("workspace",      StringType(), True),
        StructField("members_before", LongType(),   True),
        StructField("members_after",  LongType(),   True),
        StructField("delta",          LongType(),   True),
        StructField("delta_pct",      DoubleType(), True),
    ])

    df_row = spark.createDataFrame([row], schema=schema)
    _write_delta(df_row, "rls_run_summary", audit_path=audit_path)
    print(f"📝 Summary saved: {before_count} → {after_count} ({delta:+d})")
    return row


def log_membership_changes(run_id, run_timestamp, env, dataset, df_before, df_after, audit_path=None):
    """
    Compares the 'before' and 'after' snapshots of a single run and persists
    ONE ROW PER ACTUAL CHANGE (add or remove) to rls_change_log.
    """
    before_pairs = df_before.select("role_name", "upn").distinct()
    after_pairs  = df_after.select("role_name", "upn").distinct()

    added = (
        after_pairs.join(before_pairs, ["role_name", "upn"], "left_anti")
        .withColumn("operation", F.lit("add"))
    )
    removed = (
        before_pairs.join(after_pairs, ["role_name", "upn"], "left_anti")
        .withColumn("operation", F.lit("remove"))
    )

    changes = added.union(removed)
    n_changes = changes.count()

    if n_changes > 0:
        changes = (
            changes
            .withColumn("run_id", F.lit(run_id))
            .withColumn("run_timestamp", F.lit(run_timestamp))
            .withColumn("environment", F.lit(env))
            .withColumn("model", F.lit(dataset))
            .select("run_id", "run_timestamp", "environment", "model", "operation", "role_name", "upn")
        )
        _write_delta(changes, "rls_change_log", audit_path=audit_path)

    print(f"📋 Change log: {n_changes} change(s) recorded in rls_change_log")
    return n_changes


def run_with_audit(
    sync_fn, dataset, workspace, env,
    alert_threshold=0.05,
    critical_drop_threshold=0.5,
    audit_path=None,               # e.g. path_control from Connections (with trailing '/')
    alert_to=None,                 # email or list of emails for alerts
    **sync_kwargs
):
    """
    Wraps any sync function with before/after snapshots, a granular change
    log, a per-run summary, and email alerts.

    audit_path: destination ABFS prefix (e.g. path_control from Connections,
    with a trailing '/'). If left as None, tables are written to the
    notebook's default lakehouse.

    Requires 'Email notifications' to have been loaded (%run) beforehand for
    alert_to to actually send emails — otherwise alerts are only printed.

    Usage:
        run_with_audit(
            update_members_delta,
            config=config,
            dataset=dataset, workspace=workspace, env=env,
            rls=df, chunk_size=200,
            audit_path=path_control,          # from Connections
            alert_to="your-team@yourcompany.com"
        )
    """
    run_id        = str(uuid.uuid4())
    run_timestamp = datetime.utcnow().isoformat()

    before_count, df_before = snapshot_current_membership(dataset, workspace, run_id, "before", audit_path)

    result = sync_fn(dataset=dataset, workspace=workspace, **sync_kwargs)

    after_count, df_after = snapshot_current_membership(dataset, workspace, run_id, "after", audit_path)

    log_membership_changes(run_id, run_timestamp, env, dataset, df_before, df_after, audit_path)

    summary = snapshot_run_summary(dataset, workspace, env, run_id, before_count, after_count, audit_path)

    delta_pct = summary["delta_pct"] or 0

    if before_count > 0 and after_count < before_count * (1 - critical_drop_threshold):
        send_alert(
            f"Critical membership drop in '{dataset}' ({env}): "
            f"{before_count} → {after_count} ({delta_pct:.1%}). "
            f"Review before trusting this result — possible mass deletion.",
            alert_to=alert_to,
            subject=f"CRITICAL — RLS {dataset} ({env})",
            severity="critical"
        )
    elif abs(delta_pct) > alert_threshold:
        send_alert(
            f"{delta_pct:.1%} variation in membership for '{dataset}' ({env}) "
            f"(threshold: {alert_threshold:.0%}).",
            alert_to=alert_to,
            subject=f"Notice — RLS {dataset} ({env})",
            severity="warning"
        )

    return result, summary